# Zadanie 1, Część II – Text Embeddings (2 pkt) · PolEmo 2.0

Notebook odpowiada punktom z instrukcji (Text Embeddings, 2 pkt):

| Pkt zadania | Punkty | Sekcja w notebooku |
|---|---|---|
| 1. Osadzenia zdań, 2 różne modele (fastText / uśrednione Word2Vec / USE) | 0,5 | **Sekcja 1** |
| 2. t-SNE z odniesieniem do etykiet, wykresy interaktywne (hover: zdanie i anotacja) | 0,5 | **Sekcja 2** |
| 3. Analiza osadzeń i dyskusja (pytania pomocnicze 1–5) | 1,0 | **Sekcja 3** (3.1–3.5 materiał, 3.6 odpowiedzi) |

Kod - `src/text_embeddings.py`. W notebooku jest tylko uruchomienie i komentarz.
Etykiety do kolorowania: `gold` (PolEmo), `A1`, `A2` (niezależne anotacje obu iteracji) oraz `final`
(uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora, z `annotation/iterN/adjudicated.csv`).

In [1]:
import os, sys, warnings, logging, contextlib, io
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "tensorflow", "absl"):
    logging.getLogger(_n).setLevel(logging.ERROR)
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from src import text_embeddings as te

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
# Wykresy interaktywne pod komórkami: „plotly_mimetype” rysują JupyterLab, Notebook 7 i VS Code (także w niezaufanym
# notebooku), „notebook_connected” – klasyczny Notebook i eksport do HTML (plotly.js z CDN, potrzebny internet).
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Konfiguracja

In [2]:
MODELS        = ["fasttext", "use"]
ITERATIONS    = [1, 2]                # próbki z obu iteracji (po 100 zdań)
INCLUDE_BASELINE = True               # TF-IDF + SVD jako punkt odniesienia (nie liczy się do 2 modeli)
FORCE         = False                 # True = przelicz osadzenia mimo cache

FASTTEXT_PATH = None                  # None = pobierze z Hugging Face (clarin-pl/fastText-kgr10); lub ścieżka do .bin
W2V_PATH      = None                  # ścieżka do modelu z https://dsmodels.nlp.ipipan.waw.pl (wymagana dla "w2v")
USE_BACKEND   = "auto"                # "tfhub" (oryginalny USE), "distiluse" (destylat mUSE) lub "auto"

## 1. Osadzenia zdań [0,5 pkt]

**Dane:** zdania z próbek obu iteracji anotacji (`annotation/iter1`, `annotation/iter2`), etykieta `gold` z PolEmo,
anotacje A1 i A2 z obu iteracji (niezależne, przed rozstrzygnięciem konfliktów przez superanotatora;
zdania bez etykiety dostają w kolorowaniu „brak”). Etykieta `final` to uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora.

**Modele** (każdy daje jeden wektor na zdanie):
* **fastText (kgr10)** – średnia wektorów słów zdania; nieznane słowa mają wektor z n-gramów znakowych, więc
  nie ma problemu OOV (ważne przy artefaktach tokenizacji korpusu, np. „wymagał em”).
* **Word2Vec (IPI PAN)** – średnia wektorów słów; słowa spoza słownika są pomijane (raportujemy pokrycie).
* **Universal Sentence Encoder (multilingual)** – wektor całego zdania, bez uśredniania słów.
* **TF-IDF + SVD** – punkt odniesienia (worek słów), niczego nie pobiera.

In [3]:
df = te.load_sentences(ITERATIONS)
print(len(df), "zdań;", "kolumny etykiet:", te.label_columns(df))
display(df.groupby("iter")["gold"].value_counts().unstack().assign(razem=lambda x: x.sum(axis=1)))
display(df[["sent_id", "iter", "text", "gold", "A1", "A2"]].head(5))

200 zdań; kolumny etykiet: ['final', 'gold', 'A1', 'A2']


gold,AMB,NEG,NEU,POZ,razem
iter,,,,,
1,25,25,25,25,100
2,25,25,25,25,100


,sent_id,iter,text,gold,A1,A2
0,7786,1,"W obecnej chwili mam zdrową córcie , która urodziła się w terminie i rozwój jej przebiega książkowo .",NEU,NEU,NEU
1,12803,1,"Obsługa na recepcji bardzo profesjonalna , w strefie saun równiez , natomiast juz w innych miejscach ( bro...",AMB,AMB,AMB
2,23501,1,Niektóre osoby zaliczają ją do minusów gry .,NEU,NEU,NEU
3,27167,1,"Dodam tylko , że na tę wizytę czekała m 1 , 5 miesiąca .",NEG,NEG,NEU
4,18404,1,"Przeprasza że tak dość obrazowo , ale konkretnie .",AMB,NEU,NEU


**Kontrola etykiet `final`.** Poniżej liczba zdań z rozstrzygnięciem i tabela: etykieta gold z PolEmo (wiersze) kontra
uzgodniona etykieta `final` (kolumny). Wartości poza przekątną to zdania, w których zespół uznał inną etykietę niż korpus.

In [4]:
if "final" in te.label_columns(df):
    print("zdań z etykietą final:", int(df["final"].notna().sum()), "z", len(df))
    display(pd.crosstab(df["gold"], df["final"].fillna("brak"), rownames=["gold"], colnames=["final"], margins=True))
    ok = df["final"].notna()
    print(f"zgodność final z gold: {(df.loc[ok, 'final'] == df.loc[ok, 'gold']).mean():.1%}")
else:
    print("UWAGA: brak etykiet final – sprawdź annotation/iterN/adjudicated.csv (kolumny sent_id, final)")

zdań z etykietą final: 200 z 200


final,AMB,NEG,NEU,POZ,All
gold,,,,,
AMB,15,7,17,11,50
NEG,0,38,11,1,50
NEU,1,1,46,2,50
POZ,1,0,9,40,50
All,17,46,83,54,200


zgodność final z gold: 69.5%


**Jak czytać tabelę.** Wiersze to etykieta z korpusu PolEmo (`gold`), kolumny to etykieta po rozstrzygnięciu konfliktów (`final`), dla wszystkich 200 zdań z obu iteracji. Przekątna to zdania, w których korpus i zespół się zgadzają. Poza przekątną są zdania, którym zespół nadał inną etykietę niż korpus. Największa zmiana to `gold` AMB → `final` NEU, bo wg wytycznych AMB dotyczy sprzecznych ocen w jednym zdaniu, a nie ogólnego wydźwięku recenzji. To porównanie z korpusem, a nie miara zgodności anotatorów (tę liczy `python -m src.agreement`).

In [5]:
names = MODELS + (["tfidf"] if INCLUDE_BASELINE else [])
embedders, E, failed = {}, {}, {}
for n in names:
    try:
        m = te.get_embedder(n, FASTTEXT_PATH, W2V_PATH, USE_BACKEND)
        E[n] = te.get_embeddings(m, df, force=FORCE)
        embedders[n] = m
        cov = getattr(m, "last_coverage", None)
        extra = f", pokrycie słownika: {cov.mean():.1%}" if cov is not None else ""
        print(f"OK  {n:9s} {E[n].shape}  {m.label}{extra}")
    except Exception as e:
        failed[n] = f"{type(e).__name__}: {e}"
        print(f"BŁĄD {n}: {failed[n]}")
real = [n for n in E if n != "tfidf"]
if len(real) < 2:
    warnings.warn(f"Zadanie wymaga 2 modeli, działają: {real}. Sprawdź błędy powyżej.")

OK  fasttext  (200, 100)  fastText kgr10 – średnia wektorów słów
OK  use       (200, 512)  Universal Sentence Encoder multilingual (TF Hub)


OK  tfidf     (200, 100)  TF-IDF + SVD (punkt odniesienia)


### 1.3 Jak wyglądają osadzenia

Każde zdanie jest teraz **wektorem liczb**: fastText i TF-IDF dają 100 liczb, USE 512. Poniżej fragment macierzy
(pierwsze 5 zdań, pierwsze 8 wymiarów) oraz norma całego wektora. Pojedyncze wymiary nie mają interpretacji,
liczy się dopiero odległość między wektorami. Pełne osadzenia zapisują się do `results/text_embeddings/embeddings_<model>.csv`.

In [6]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    print(f"macierz: {E[n].shape[0]} zdań × {E[n].shape[1]} wymiarów, dtype {E[n].dtype}, "
          f"średnia norma wektora: {np.linalg.norm(E[n], axis=1).mean():.2f}")
    display(te.embedding_preview(df, E[n], n_rows=5, n_dims=8))
    print("zapisano:", te.export_embeddings_csv(df, E[n], n))


=== fastText kgr10 – średnia wektorów słów ===
macierz: 200 zdań × 100 wymiarów, dtype float32, średnia norma wektora: 28.92


,sent_id,zdanie,d0,d1,d2,d3,d4,d5,d6,d7,… (+92),norma
0,7786,"W obecnej chwili mam zdrową córcie , która urodziła się…",-0.334,3.241,-0.811,1.834,4.347,3.953,-3.335,7.373,…,29.049999
1,12803,"Obsługa na recepcji bardzo profesjonalna , w strefie sa…",-2.224,2.751,0.497,0.439,2.908,2.459,-2.838,6.211,…,24.809999
2,23501,Niektóre osoby zaliczają ją do minusów gry .,-1.980,2.700,2.732,1.349,2.510,-2.089,-1.667,2.215,…,32.189999
3,27167,"Dodam tylko , że na tę wizytę czekała m 1 , 5 miesiąca …",-5.607,2.404,-0.601,-1.845,4.473,2.186,1.159,3.014,…,27.879999
4,18404,"Przeprasza że tak dość obrazowo , ale konkretnie .",-5.957,0.599,-0.126,1.948,2.925,-0.521,-3.929,0.349,…,33.180000


zapisano: /mnt/c/Users/barte/Desktop/nlp/results/text_embeddings/embeddings_fasttext.csv

=== Universal Sentence Encoder multilingual (TF Hub) ===
macierz: 200 zdań × 512 wymiarów, dtype float32, średnia norma wektora: 1.00


,sent_id,zdanie,d0,d1,d2,d3,d4,d5,d6,d7,… (+504),norma
0,7786,"W obecnej chwili mam zdrową córcie , która urodziła się…",0.014,-0.002,0.018,0.002,-0.094,-0.047,0.009,-0.009,…,1.0
1,12803,"Obsługa na recepcji bardzo profesjonalna , w strefie sa…",0.000,-0.005,0.014,0.031,0.010,0.061,-0.073,0.037,…,1.0
2,23501,Niektóre osoby zaliczają ją do minusów gry .,-0.031,0.081,-0.041,0.000,0.053,-0.014,-0.004,0.001,…,1.0
3,27167,"Dodam tylko , że na tę wizytę czekała m 1 , 5 miesiąca …",-0.020,0.014,-0.046,-0.002,-0.113,0.022,-0.010,0.070,…,1.0
4,18404,"Przeprasza że tak dość obrazowo , ale konkretnie .",-0.047,-0.013,0.091,-0.014,0.085,-0.026,-0.014,0.021,…,1.0


zapisano: /mnt/c/Users/barte/Desktop/nlp/results/text_embeddings/embeddings_use.csv

=== TF-IDF + SVD (punkt odniesienia) ===
macierz: 200 zdań × 100 wymiarów, dtype float64, średnia norma wektora: 0.74


,sent_id,zdanie,d0,d1,d2,d3,d4,d5,d6,d7,… (+92),norma
0,7786,"W obecnej chwili mam zdrową córcie , która urodziła się…",0.124,-0.077,0.038,-0.044,-0.091,0.003,0.112,0.091,…,0.75
1,12803,"Obsługa na recepcji bardzo profesjonalna , w strefie sa…",0.116,-0.060,0.097,-0.013,0.065,-0.026,0.110,0.079,…,0.72
2,23501,Niektóre osoby zaliczają ją do minusów gry .,0.047,-0.050,-0.028,-0.036,-0.041,0.039,-0.015,0.008,…,0.77
3,27167,"Dodam tylko , że na tę wizytę czekała m 1 , 5 miesiąca …",0.092,-0.002,-0.052,-0.075,0.044,-0.087,-0.116,-0.069,…,0.74
4,18404,"Przeprasza że tak dość obrazowo , ale konkretnie .",0.061,0.035,-0.021,-0.057,0.077,-0.059,0.004,-0.028,…,0.76


zapisano: /mnt/c/Users/barte/Desktop/nlp/results/text_embeddings/embeddings_tfidf.csv


**Podobieństwo kosinusowe wybranych zdań.** Po dwa zdania z każdej klasy gold (iteracja 1). Jasne pola = zdania bliskie
w danej przestrzeni, na przekątnej zawsze 1. w fastText wszystkie zdania są do siebie podobne,
w USE i TF-IDF większość par jest bliska zera.

In [7]:
sel = (df[df["iter"] == df["iter"].min()].groupby("gold", sort=False).head(2)).sort_values("gold").index
for n in E:
    te.similarity_heatmap(df, E[n], sel, f"Podobieństwo kosinusowe – {embedders[n].label}").show()

## 2. Wizualizacja t-SNE z odniesieniem do etykiet [0,5 pkt]

Wykres jest interaktywny: **po najechaniu na punkt widać pełne zdanie oraz etykiety** (gold, A1, A2, final),
iterację i `sent_id`. Domyślnie punkty są kolorowane **uzgodnioną anotacją (`final`)**, a lista rozwijana nad wykresem przełącza kolorowanie między źródłami etykiet,
kliknięcie w legendę ukrywa klasę. Kółko = iteracja 1, romb = iteracja 2.
t-SNE: odległość kosinusowa, inicjalizacja PCA, `perplexity = min(30, (n−1)/3)`, stałe ziarno.
Wykresy zapisują się też jako HTML w `results/text_embeddings/`

In [8]:
xy = {}
for n in E:
    xy[n] = te.tsne_2d(E[n])
    fig = te.make_tsne_figure(df, xy[n], f"t-SNE – {embedders[n].label}")
    fig.write_html(te.OUT_DIR / f"tsne_{n}.html", include_plotlyjs="cdn")
    fig.show()

**Jak czytać wykresy:** jeśli klasy wydźwięku tworzą oddzielne skupiska, osadzenie koduje sygnał przydatny
do klasyfikacji. Jeśli POZ i NEG są wymieszane, a skupiska mogą odpowiadać np. tematowi (hotel / lekarz /
szkoła / produkt) lub długości zdania, model koduje głównie temat.

## 3. Analiza osadzeń i dyskusja [1 pkt]

odpowiedzi w 3.6

### 3.1 Pyt. 1 – zdania uznane za podobne, a różniące się znaczeniem
Pary zdań o przeciwnym wydźwięku (POZ vs NEG według gold) z najwyższym podobieństwem kosinusowym.

In [9]:
sims = {n: te.cosine_matrix(E[n]) for n in E}
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.similar_but_different(df, sims[n], k=8))


=== fastText kgr10 – średnia wektorów słów ===


,sent_id_a,tekst_a,gold_a,sent_id_b,tekst_b,gold_b,cosine
0,44042,"Oceniam Panią Doktor niziutko , w moim przypadku ( refluks żołądkowo - pzrełykowy ) nie udało się doprowad...",NEG,21082,Ujęło mnie w Pani Doktor indywidualne podejście do problemu pacjenta .,POZ,0.940
1,44042,"Oceniam Panią Doktor niziutko , w moim przypadku ( refluks żołądkowo - pzrełykowy ) nie udało się doprowad...",NEG,38629,"Pan Bartłomiej wysłuchał długiej historii mojego urazu zadając przy tym pytania o istotne szczegóły , przy...",POZ,0.929
2,44042,"Oceniam Panią Doktor niziutko , w moim przypadku ( refluks żołądkowo - pzrełykowy ) nie udało się doprowad...",NEG,25566,"Kilka sytuacji w trakcie mojej ciąży potwierdziły , że jest to wyjątkowy lekarz , taki , jakich się raczej...",POZ,0.919
3,14630,"Nie wiem jak można pracować na oddziale , TAKIM oddziale i nie mieć dla ludzi ani trochę szacunku , współc...",NEG,20436,"Pani Barbara na pewno dobrze wyłozy sprawę , dobrze opowie i każdy wie .",POZ,0.915
4,41441,Pierwszy kurs językowy i pozytywne zaskoczenie : ) Pani naprawdę zna sie na tym co robi i ma odpowiednie p...,POZ,8734,"Myśle , że lekarz powinien dokładnie wysłuchać pacjenta , co ten ma do powiedzenia na temat swoich dolegli...",NEG,0.911
5,14630,"Nie wiem jak można pracować na oddziale , TAKIM oddziale i nie mieć dla ludzi ani trochę szacunku , współc...",NEG,38629,"Pan Bartłomiej wysłuchał długiej historii mojego urazu zadając przy tym pytania o istotne szczegóły , przy...",POZ,0.910
6,25566,"Kilka sytuacji w trakcie mojej ciąży potwierdziły , że jest to wyjątkowy lekarz , taki , jakich się raczej...",POZ,41387,"Pani dr zachowywała się jakby mnie wcześniej nie znała w ogóle , była bardzo nie miła , opryskliwa , wchod...",NEG,0.907
7,41441,Pierwszy kurs językowy i pozytywne zaskoczenie : ) Pani naprawdę zna sie na tym co robi i ma odpowiednie p...,POZ,14630,"Nie wiem jak można pracować na oddziale , TAKIM oddziale i nie mieć dla ludzi ani trochę szacunku , współc...",NEG,0.905



=== Universal Sentence Encoder multilingual (TF Hub) ===


,sent_id_a,tekst_a,gold_a,sent_id_b,tekst_b,gold_b,cosine
0,38629,"Pan Bartłomiej wysłuchał długiej historii mojego urazu zadając przy tym pytania o istotne szczegóły , przy...",POZ,8734,"Myśle , że lekarz powinien dokładnie wysłuchać pacjenta , co ten ma do powiedzenia na temat swoich dolegli...",NEG,0.483
1,25566,"Kilka sytuacji w trakcie mojej ciąży potwierdziły , że jest to wyjątkowy lekarz , taki , jakich się raczej...",POZ,8734,"Myśle , że lekarz powinien dokładnie wysłuchać pacjenta , co ten ma do powiedzenia na temat swoich dolegli...",NEG,0.474
2,44042,"Oceniam Panią Doktor niziutko , w moim przypadku ( refluks żołądkowo - pzrełykowy ) nie udało się doprowad...",NEG,21082,Ujęło mnie w Pani Doktor indywidualne podejście do problemu pacjenta .,POZ,0.474
3,33410,Można zostawić bagaże po wymeldowaniu .,POZ,12988,Można po nim się pochorować .,NEG,0.449
4,13554,"Od dwóch miesięcy lekarze próbują ustalić , co mi dolega robiąc szczegółowe badania , wreszcie neurolog za...",NEG,21082,Ujęło mnie w Pani Doktor indywidualne podejście do problemu pacjenta .,POZ,0.431
5,19496,Jednak to co zobaczyli śmy w środku było dosyć przerażajace .,NEG,3579,"Mieli śmy widok za morze , więc to też plus .",POZ,0.430
6,13554,"Od dwóch miesięcy lekarze próbują ustalić , co mi dolega robiąc szczegółowe badania , wreszcie neurolog za...",NEG,11439,"Pani Doktor sprawia wrazenie rzeczowej , milej i symaptycznej .",POZ,0.418
7,16311,"Kobieta zauważyła , że jestem w b . złym stanie psychicznym , podała coś , co jeszcze poprawiło mi nastrój .",POZ,41387,"Pani dr zachowywała się jakby mnie wcześniej nie znała w ogóle , była bardzo nie miła , opryskliwa , wchod...",NEG,0.416



=== TF-IDF + SVD (punkt odniesienia) ===


,sent_id_a,tekst_a,gold_a,sent_id_b,tekst_b,gold_b,cosine
0,21728,Za to go podziwiam .,POZ,44879,LG powinna zwracać za to pieniądze .,NEG,0.752
1,30989,"Nigdy nie wiedzieli śmy , co akurat danego dnia zostanie podane .",NEG,20222,"ALE , z kontaktem telefonicznym NIGDY nie było problemów .",POZ,0.617
2,33410,Można zostawić bagaże po wymeldowaniu .,POZ,12988,Można po nim się pochorować .,NEG,0.588
3,36550,"Pokoje bardzo przyjemne , każdy wyposażony jest w czajnik elektryczny , chłodziarkę i suszarkę do włosów (...",POZ,12359,Wybrał em te słuchawki ze względu na ( pozornie ) świetną jakość wykonania i jakość brzmienia .,NEG,0.499
4,24190,"Dr Szuber to świetna lekarka , bardzo polecam !",POZ,36137,Masakra nie polecam tego miejsca ! ! !,NEG,0.434
5,34110,Przy poziomie Gołębiewskiego to rudera .,NEG,10075,"Wszystko czego moglabym oczekiwac , zostalo spelnione na najwyzszym poziomie .",POZ,0.409
6,45015,"Jak mi powiedzial , ze bakteria , ktora mam jest smiertelna , cos mi zaswitalo w glowie i dzieki pomocy zy...",NEG,24190,"Dr Szuber to świetna lekarka , bardzo polecam !",POZ,0.388
7,19041,Wzięła m receptę i rozpłakała m się po wyjściu z gabinetu .,NEG,31645,"Dzięki terapii przekonała m się , że ode mnie zależy jak się czuję .",POZ,0.385


### 3.2 Pyt. 2 – zdania podobne znaczeniowo, a uznane za odległe (inne słowa)
Dwie metody. **(a)** Pary, które jeden model uznaje za bliskie, a drugi za odległe (rangi percentylowe wszystkich par) – to kandydaci, trzeba je ocenić ręcznie. **(b)** Kontrolowane pary: parafrazy (inne słowa, ten sam sens) vs. kontrola niepowiązana.

In [10]:
if len(real) >= 2:
    a, b = real[0], real[1]
else:
    a, b = list(E)[:2]
if len(E) >= 2:
    hi_a, hi_b = te.cross_model_disagreement(df, sims[a], sims[b], a, b, k=8)
    print(f"Blisko według {a}, daleko według {b}:"); display(hi_a)
    print(f"Blisko według {b}, daleko według {a}:"); display(hi_b)

Blisko według fasttext, daleko według use:


,sent_id_a,tekst_a,gold_a,sent_id_b,tekst_b,gold_b,ranga_fasttext,ranga_use
0,36880,Nauczyłąm sie tak zadawać pytania na szkoleniach dla handlowców żeby uzyskać konkretną odpowiedź a nie zda...,NEU,42528,Podobnie może być i w przypadku dźwięku - przepuszczenie dźwięków przez pewne filtry i cyfrowe ich przetwo...,NEU,0.992,0.019
1,17758,"Czystość pozostawia wiele do życzenia ( paprochy na wykładzinie ) , jakość obsługi w większości przypadków...",AMB,3480,"Można wybrać się klifami na ścieżkę edukacyjną w stronę latarni morskiej , albo dojść do drewnianego pomos...",POZ,0.979,0.009
2,23214,"Badaczki z Wydziału Biologii i Biotechnologii UMCS postanowiły sprawdzić , czy enzymu tego nie można wykor...",NEU,38201,"Warto zgłaszać się do tablicy , plusy będą bardzo potrzebne przy zaliczeniu .",AMB,0.985,0.018
3,17224,"Podsumowując hotel z bardzo dużym potencjałem jednak wydaje mi się , że zarządzanie nim raczkuje i jest na...",AMB,9779,Nie uczcie się tylko z instrukcji ! ! ! !,AMB,0.961,0.003
4,45015,"Jak mi powiedzial , ze bakteria , ktora mam jest smiertelna , cos mi zaswitalo w glowie i dzieki pomocy zy...",NEG,32230,Pieniedzy oczywiscie nie chca wrocic,NEG,0.958,0.004
5,36880,Nauczyłąm sie tak zadawać pytania na szkoleniach dla handlowców żeby uzyskać konkretną odpowiedź a nie zda...,NEU,16488,Faktycznie – było to widoczne ( wystarczyło kazać stanąć dziecku prosto ) .,NEU,0.976,0.026
6,32237,"Owe kolokwia składają się z zadań trudniejszych niż w skrypcie , lecz jeśli się uczy systematycznie , to m...",AMB,41123,"Po prośbie wyjaśnienia przez recepcjonistów , każdy z nich mówił co innego .",NEG,0.966,0.020
7,26785,Nawet nie zerknął na zabandożowaną dłoń .,NEG,20436,"Pani Barbara na pewno dobrze wyłozy sprawę , dobrze opowie i każdy wie .",POZ,0.985,0.042


Blisko według use, daleko według fasttext:


,sent_id_a,tekst_a,gold_a,sent_id_b,tekst_b,gold_b,ranga_fasttext,ranga_use
0,26170,U Pana Doktora pierwszy raz była m w listopadzie .,NEU,39368,A było to 2 miesiące temu .,NEU,0.014,0.992
1,26170,U Pana Doktora pierwszy raz była m w listopadzie .,NEU,28605,Używam jej od ok 2 lat .,NEU,0.016,0.985
2,5343,Kontrola za 2 miesiące .,NEU,13554,"Od dwóch miesięcy lekarze próbują ustalić , co mi dolega robiąc szczegółowe badania , wreszcie neurolog za...",NEG,0.021,0.985
3,5532,W rozmowie pan miły .,POZ,11535,Syf to mało powiedziane .,NEG,0.031,0.995
4,21728,Za to go podziwiam .,POZ,28605,Używam jej od ok 2 lat .,NEU,0.002,0.959
5,41824,"To nie jest hotel 4 * , max . 2 * .",NEG,36137,Masakra nie polecam tego miejsca ! ! !,NEG,0.030,0.986
6,41824,"To nie jest hotel 4 * , max . 2 * .",NEG,17224,"Podsumowując hotel z bardzo dużym potencjałem jednak wydaje mi się , że zarządzanie nim raczkuje i jest na...",AMB,0.013,0.968
7,11439,"Pani Doktor sprawia wrazenie rzeczowej , milej i symaptycznej .",POZ,30467,"Była m u pani doktor ok . 2 lata temu , to była moja pierwsza wizyta u ginekologa .",NEU,0.043,0.998


In [11]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    probes = te.probe_table(embedders)      # modele ładują się z dysku; komunikaty ukryte
display(probes)
print("Średnie cos. wg kategorii:")
display(probes.groupby("kategoria", sort=False)[list(E)].mean().round(3))

I0000 00:00:1792500891.181596   22196 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1792500896.331608   22196 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


,kategoria,zdanie_a,zdanie_b,fasttext,use,tfidf
0,parafraza,Obsługa była bardzo miła i pomocna .,Personel okazał się niezwykle uprzejmy i życzliwy .,0.681,0.340,0.284
1,parafraza,Pokój był czysty i dobrze wyposażony .,"W pokoju panowała czystość , a wyposażenie było kompletne .",0.688,0.613,0.297
2,parafraza,Lekarz dokładnie mnie zbadał i wszystko wytłumaczył .,Doktor przeprowadził staranne badanie i cierpliwie objaśnił wyniki .,0.868,0.524,0.547
3,"przeciwny sens, podobne słowa",Polecam ten hotel każdemu .,Nie polecam tego hotelu nikomu .,0.619,0.607,0.433
4,"przeciwny sens, podobne słowa",Jedzenie było smaczne .,Jedzenie było niesmaczne .,0.981,0.803,0.966
5,"przeciwny sens, podobne słowa",Obsługa była bardzo miła .,Obsługa była bardzo niemiła .,0.978,0.973,0.832
6,"przeciwny sens, te same słowa","Obsługa była świetna , ale pokój fatalny .","Pokój był świetny , ale obsługa fatalna .",0.892,0.924,0.770
7,"ten sam sens, różna długość",Polecam .,"Szczerze polecam ten hotel wszystkim znajomym i rodzinie , bo spędziliśmy tam naprawdę udany tydzień .",0.662,0.078,0.452
8,"ten sam sens, różna długość",Bardzo słaby serwis .,"Obsługa w tym miejscu była naprawdę bardzo słaba i musieliśmy długo czekać na wszystko , co zamówiliśmy .",0.602,0.264,0.419
9,"ten sam sens, różna długość",Wszystko w porządku .,"Pobyt przebiegł bez żadnych problemów , wszystko było tak , jak należy , i nie mamy żadnych zastrzeżeń .",0.549,0.232,0.297


Średnie cos. wg kategorii:


,fasttext,use,tfidf
kategoria,,,
parafraza,0.746,0.492,0.376
"przeciwny sens, podobne słowa",0.859,0.794,0.744
"przeciwny sens, te same słowa",0.892,0.924,0.770
"ten sam sens, różna długość",0.604,0.191,0.389
kontrola: niepowiązane,0.506,0.111,0.004


Zdania w tabeli powyżej nie pochodzą z korpusu. to zestaw kontrolny napisany ręcznie (`PROBE_PAIRS` w `src/text_embeddings.py`), zrobiony po to, żeby sprawdzić zjawiska z pytań 1–3, które w 200 losowych zdaniach trudno znaleźć. Kolumna `kategoria` mówi, jaka jest relacja między a i b: `parafraza` (ten sam sens, inne słowa), `przeciwny sens, podobne słowa` („smaczne” / „niesmaczne”), `przeciwny sens, te same słowa` (te same słowa w innej kolejności), `ten sam sens, różna długość` oraz `kontrola: niepowiązane` (linia bazowa: ile wynosi cosinus „z niczego”). Liczby w kolumnach modeli to cosinus między wektorami obu zdań. Dla 12 par wnioski są ilustracją, a nie statystyką; dane z korpusu są w 3.1 i 3.2(a).

### 3.3 Pyt. 3 – różna długość, podobny sens
Kategoria „ten sam sens, różna długość” w tabeli powyżej (porównaj z parafrazami i z kontrolą niepowiązaną).

In [12]:
len_pairs = probes[probes["kategoria"].str.contains("różna długość")].copy()
len_pairs["dł_a"] = len_pairs["zdanie_a"].str.split().str.len()
len_pairs["dł_b"] = len_pairs["zdanie_b"].str.split().str.len()
display(len_pairs[["zdanie_a", "zdanie_b", "dł_a", "dł_b"] + list(E)])

,zdanie_a,zdanie_b,dł_a,dł_b,fasttext,use,tfidf
7,Polecam .,"Szczerze polecam ten hotel wszystkim znajomym i rodzinie , bo spędziliśmy tam naprawdę udany tydzień .",2,16,0.662,0.078,0.452
8,Bardzo słaby serwis .,"Obsługa w tym miejscu była naprawdę bardzo słaba i musieliśmy długo czekać na wszystko , co zamówiliśmy .",4,18,0.602,0.264,0.419
9,Wszystko w porządku .,"Pobyt przebiegł bez żadnych problemów , wszystko było tak , jak należy , i nie mamy żadnych zastrzeżeń .",4,19,0.549,0.232,0.297


### 3.4 Pyt. 4 – długość zdania a wartości podobieństwa i „wyrazistość” wektora
Przedziały długości (kwantyle liczby tokenów): norma surowego wektora (dla uśrednianych wektorów słów to miara siły sygnału; dla USE norma jest zwykle ≈ stała, więc nie ma interpretacji), cosinus do centroidu korpusu i średnie podobieństwo do pozostałych zdań.

In [13]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.length_stats(df, E[n]))

fig = go.Figure()
for n in E:
    S = sims[n].copy(); np.fill_diagonal(S, np.nan)
    fig.add_trace(go.Scatter(x=df["n_tokens"], y=np.nanmean(S, axis=1), mode="markers", name=n,
                             opacity=0.6, text=df["text"], hovertemplate="%{text}<br>tokenów: %{x}, śr. cos: %{y:.3f}<extra>"+n+"</extra>"))
fig.update_layout(title="Długość zdania vs średnie podobieństwo do pozostałych zdań", template="plotly_white",
                  xaxis_title="liczba tokenów", yaxis_title="średni cosinus do innych zdań", height=520)
fig.show()


=== fastText kgr10 – średnia wektorów słów ===


,n,norma_wektora,cos_do_centroidu,srednie_cos_do_innych
bin,,,,
"(4.999, 11.333]",67.0,30.301001,0.763,0.624
"(11.333, 20.0]",70.0,28.617001,0.809,0.662
"(20.0, 36.0]",63.0,27.796000,0.893,0.731
"Spearman(n_tokens, śr. cos)",NaN,NaN,NaN,0.588



=== Universal Sentence Encoder multilingual (TF Hub) ===


,n,norma_wektora,cos_do_centroidu,srednie_cos_do_innych
bin,,,,
"(4.999, 11.333]",67.0,1.0,0.147,0.024
"(11.333, 20.0]",70.0,1.0,0.214,0.038
"(20.0, 36.0]",63.0,1.0,0.235,0.042
"Spearman(n_tokens, śr. cos)",NaN,NaN,NaN,0.318



=== TF-IDF + SVD (punkt odniesienia) ===


,n,norma_wektora,cos_do_centroidu,srednie_cos_do_innych
bin,,,,
"(4.999, 11.333]",67.0,0.755,0.120,0.014
"(11.333, 20.0]",70.0,0.739,0.153,0.019
"(20.0, 36.0]",63.0,0.740,0.192,0.025
"Spearman(n_tokens, śr. cos)",NaN,NaN,NaN,0.589


Zdania podzielono na 3 grupy wg liczby tokenów, tak aby w grupach było podobnie dużo zdań.

* `n` – liczba zdań w grupie.
* `norma_wektora` – średnia długość surowego wektora zdania. W fastText wektor zdania to średnia wektorów słów, więc w dłuższych zdaniach wektory częściowo się znoszą i norma maleje. W USE norma jest stała (1), więc ta kolumna nic nie mówi.
* `cos_do_centroidu` – podobieństwo zdania do środka (średniej) wszystkich zdań. Im wyżej, tym zdanie jest „bardziej przeciętne”.
* `srednie_cos_do_innych` – średnie podobieństwo zdania do wszystkich pozostałych.
* Ostatni wiersz – współczynnik Spearmana między liczbą tokenów a średnim cosinusem. Dodatni znak = im dłuższe zdanie, tym bardziej podobne do reszty.

Wniosek: w fastText długie zdania zbliżają się do średniego wektora, więc wysokie cosinusy mogą wynikać z długości, a nie z podobnego sensu.

### 3.5 Pyt. 5 – czy osadzenia nadają się do zadania (klasyfikacja wydźwięku zdania)?
Regresja logistyczna (walidacja krzyżowa warstwowa) na samych osadzeniach, czystość k najbliższych sąsiadów względem poziomu losowego i silhouette. **Punkt odniesienia dla accuracy to `baza_wiekszosciowa`** (udział najliczniejszej klasy, czyli wynik klasyfikatora zawsze wybierającego tę klasę); `losowy_wg_rozkladu` to losowanie zgodne z rozkładem klas, a `acc_minus_baza` mówi, o ile model go przekracza. Etykiety (gold, A1, A2, final) są liczone na zdaniach, które je mają. `final` to uzgodnione etykiety, więc to ją traktujemy jako główny punkt odniesienia; gold PolEmo i A1/A2 służą do porównania. Przy ~200 zdaniach wyniki są orientacyjne, nie ma tu przedziałów ufności.

In [14]:
rows = []
for n in E:
    for col in te.label_columns(df):
        r = te.evaluate_embeddings(E[n], df[col]); r.update(model=n, etykiety=col); rows.append(r)
res = pd.DataFrame(rows).set_index(["model", "etykiety"])
display(res)

n  accuracy  macro_F1  kNN10_czystosc  losowy_wg_rozkladu  baza_wiekszosciowa  silhouette_cos  acc_minus_baza
model    etykiety                                                                                                                 
fasttext final     200     0.505     0.322           0.321               0.305               0.415          -0.123           0.090
         gold      200     0.395     0.388           0.282               0.250               0.250          -0.072           0.145
         A1        200     0.525     0.341           0.330               0.312               0.430          -0.117           0.095
         A2        200     0.495     0.291           0.304               0.308               0.420          -0.147           0.075
use      final     200     0.520     0.330           0.334               0.305               0.415          -0.013           0.105
         gold      200     0.465     0.464           0.321               0.250               0.250          -0.003           0.215
         A1        200     0.545     0.342           0.350               0.312               0.430          -0.013           0.115
         A2        200     0.540     0.364           0.336               0.308               0.420          -0.014           0.120
tfidf    final     200     0.490     0.318           0.335               0.305               0.415          -0.008           0.075
         gold      200     0.375     0.374           0.291               0.250               0.250          -0.006           0.125
         A1        200     0.485     0.299           0.356               0.312               0.430          -0.008           0.055
         A2        200     0.430     0.266           0.330               0.308               0.420          -0.009           0.010

Klasyfikator (regresja logistyczna) przewiduje wydźwięk zdania z samego wektora zdania, czyli sprawdzamy jakość osadzeń, a nie samego klasyfikatora. Wiersze to model osadzeń (`fasttext`, `use`, `tfidf` jako punkt odniesienia) × zestaw etykiet, które traktujemy jako prawdę (`final`, `gold`, `A1`, `A2`). Wynik jest z 5-krotnej walidacji krzyżowej, więc klasyfikator nie ocenia zdań, na których się uczył.

* `accuracy` – odsetek poprawnych przewidywań. Porównuj ją z `baza_wiekszosciowa` (wynik klasyfikatora, który zawsze wybiera najliczniejszą klasę); `acc_minus_baza` to różnica. Przy `final` i A1/A2 klasa NEU jest najliczniejsza, więc same accuracy bywa mylące.
* `macro_F1` – średnia F1 po klasach, każda klasa liczy się tak samo, więc ignorowanie rzadkich klas (AMB) mocno obniża wynik.
* `kNN10_czystosc` – średni odsetek 10 najbliższych sąsiadów zdania (wg cosinusa) z tą samą etykietą. Nie używa klasyfikatora, mierzy samą geometrię osadzenia. Odnosi się do `losowy_wg_rozkladu` (losowanie zgodne z rozkładem klas).
* `silhouette_cos` – zwykle miara klasteryzacji, ale działa dla dowolnego podziału na grupy, także dla etykiet. Pokazuje, czy zdania jednej klasy leżą bliżej siebie niż zdań innych klas (od −1 do 1). Wartości ≤ 0 to brak oddzielnych skupisk, zgodnie z t-SNE.
* Różnice między `gold` a `final`/A1/A2: `gold` jest zbalansowane (po 50 zdań na klasę), a w `final` i A1/A2 dominuje NEU, więc accuracy jest wyższa głównie dlatego, że łatwiej trafić w NEU.

Macierz pomyłek (wiersze = uzgodniona etykieta `final`, kolumny = predykcja klasyfikatora na samych
osadzeniach) i precyzja / czułość / F1 dla każdej klasy. Pokazuje, czy problem leży w klasach „środkowych” (NEU, AMB), czy też
osadzenia mylą nawet POZ z NEG.

In [15]:
main_col = "final" if "final" in te.label_columns(df) else "gold"
print("etykiety:", main_col)
for n in E:
    cm, rep = te.confusion_report(E[n], df[main_col])
    print(f"\n=== {embedders[n].label} ===")
    display(cm); display(rep)

etykiety: final

=== fastText kgr10 – średnia wektorów słów ===


predykcja,POZ,NEG,NEU,AMB
etykieta,,,,
POZ,18,6,30,0
NEG,7,7,32,0
NEU,2,5,76,0
AMB,6,1,10,0


,precision,recall,f1-score,support
POZ,0.545,0.333,0.414,54.000
NEG,0.368,0.152,0.215,46.000
NEU,0.514,0.916,0.658,83.000
AMB,0.000,0.000,0.000,17.000
accuracy,0.505,0.505,0.505,0.505
macro avg,0.357,0.350,0.322,200.000
weighted avg,0.445,0.505,0.434,200.000



=== Universal Sentence Encoder multilingual (TF Hub) ===


predykcja,POZ,NEG,NEU,AMB
etykieta,,,,
POZ,27,8,19,0
NEG,10,4,32,0
NEU,7,3,73,0
AMB,8,2,7,0


,precision,recall,f1-score,support
POZ,0.519,0.500,0.509,54.00
NEG,0.235,0.087,0.127,46.00
NEU,0.557,0.880,0.682,83.00
AMB,0.000,0.000,0.000,17.00
accuracy,0.520,0.520,0.520,0.52
macro avg,0.328,0.367,0.330,200.00
weighted avg,0.426,0.520,0.450,200.00



=== TF-IDF + SVD (punkt odniesienia) ===


predykcja,POZ,NEG,NEU,AMB
etykieta,,,,
POZ,17,3,34,0
NEG,5,8,33,0
NEU,7,3,73,0
AMB,4,2,11,0


,precision,recall,f1-score,support
POZ,0.515,0.315,0.391,54.00
NEG,0.500,0.174,0.258,46.00
NEU,0.483,0.880,0.624,83.00
AMB,0.000,0.000,0.000,17.00
accuracy,0.490,0.490,0.490,0.49
macro avg,0.375,0.342,0.318,200.00
weighted avg,0.455,0.490,0.424,200.00


Wiersz to prawdziwa etykieta (`final`), kolumna to to, co przewidział klasyfikator. Liczby na przekątnej to trafienia. Jeśli prawie wszystko ląduje w kolumnie NEU, klasyfikator „ucieka” do najliczniejszej klasy. Zwróć uwagę na `recall` NEG i AMB: bliskie 0 oznacza, że osadzenia prawie nie odróżniają tych klas od NEU.

### 3.6 Odpowiedzi na pytania pomocnicze

Każda odpowiedź ma opierać się na konkretnych przykładach i liczbach z sekcji 3.1–3.5.

**1. Czy są zdania, które model uznał za podobne, ale w rzeczywistości różnią się znaczeniem?**
_Odpowiedź:_ Tak. W zestawie kontrolnym (ręcznie napisane pary) zdania o przeciwnym sensie i podobnych słowach mają wyższy średni cosinus niż prawdziwe parafrazy: 0,859 wobec 0,746 w fastText oraz 0,794 wobec 0,492 w USE. Na przykład "Obsługa była bardzo miła" i "bardzo niemiła" dają 0,978 w fastText i 0,973 w USE, a "smaczne" i "niesmaczne" 0,981 (fastText) i 0,803 (USE). Para "Obsługa była świetna, ale pokój fatalny" i "Pokój był świetny, ale obsługa fatalna" ma 0,892 (fastText) i 0,924 (USE), bo oba zdania zawierają te same słowa w innej kolejności.

W korpusie widać to samo: najbliższe pary POZ i NEG w fastText (cosinus 0,90–0,94) dotyczą tego samego tematu, czyli lekarza i pacjenta. Modele kodują więc głównie temat i słownictwo, a nie wydźwięk. W fastText zdanie reprezentujemy jako średnią wektorów jego słów, więc negacja "nie" ma mały wpływ na wynik, a zdania przeciwne w sensie wychodzą bardzo podobne (np. "miła" i "niemiła" 0,978). USE, który koduje całe zdanie, też wyraźnie nie rozróżnia takich par.

**2. Czy są zdania semantycznie podobne, ale model uznał je za odległe, bo użyto innych słów?**
_Odpowiedź:_ Tak. Parafrazy z zestawu kontrolnego (ręcznie napisane pary) mają niski średni cosinus: 0,746 w fastText, 0,492 w USE i 0,376 w TF-IDF. Na przykład "Obsługa była bardzo miła i pomocna" i "Personel okazał się niezwykle uprzejmy i życzliwy" dają 0,681 (fastText), 0,340 (USE) i 0,284 (TF-IDF). TF-IDF zawodzi najbardziej, bo widzi tylko wspólne słowa. USE ma niskie wartości bezwzględne, ale w swojej skali wyraźnie odróżnia parafrazy od zdań niepowiązanych (0,492 wobec 0,111).

W tabeli rozbieżności między modelami (3.2a) widać, że fastText uznaje za bliskie zdania z różnych tematów (rangi percentylowe 0,96–0,99), np. pytania zadawane na szkoleniach handlowców i przepuszczanie dźwięku przez filtry. Wynika to z tego, że w fastText wszystkie zdania są do siebie podobne (średni cosinus niepowiązanych par to aż 0,506). USE poprawnie zbliża zdania o czasie ("2 miesiące temu" i "od dwóch miesięcy"), ale myli się przy "W rozmowie pan miły" i "Syf to mało powiedziane" (ranga 0,995), czyli przy dwóch krótkich ocenach o przeciwnym wydźwięku.

**3. Czy zdania o różnej długości, ale podobnym sensie, są traktowane przez model podobnie?**
_Odpowiedź:_ Nie w pełni. W zestawie kontrolnym (ręcznie napisane pary) krótkie zdanie i jego długa wersja o tym samym sensie mają niski średni cosinus: 0,604 w fastText, 0,191 w USE i 0,389 w TF-IDF. To mniej niż dla parafraz o podobnej długości (0,746, 0,492 i 0,376 w kolejności fastText, USE, TF-IDF). W fastText wynik jest tylko trochę wyższy niż dla zdań niepowiązanych (0,506), a w USE niewiele (0,191 wobec 0,111), więc modele słabo odróżniają takie pary od zdań bez związku.  Na przykład "Polecam ." (2 tokeny) i "Szczerze polecam ten hotel wszystkim znajomym i rodzinie, bo spędziliśmy tam naprawdę udany tydzień ." (16 tokenów) mają cosinus 0,662 (fastText), 0,078 (USE) i 0,452 (TF-IDF). Para "Wszystko w porządku ." i długie zdanie o pobycie bez zastrzeżeń daje 0,549 (fastText) i 0,232 (USE). Długie zdanie wnosi dużo dodatkowej treści (hotel, rodzina, tydzień), więc jego wektor oddala się od wektora krótkiej oceny. 

**4. Czy zauważasz różnicę w wartościach podobieństwa między krótkimi a długimi zdaniami? Czy długość wpływa na „wyrazistość” wektora?**
_Odpowiedź:_ Tak. Zdania zostały podzielone na trzy grupy wg liczby tokenów (słowa i interpunkcja): krótkie (5–11 tokenów, 67 zdań), średnie (12–20, 70 zdań) i długie (21–36, 63 zdania). W fastText podobieństwo rośnie wraz z długością: cosinus do centroidu (środka wszystkich zdań) wynosi 0,763, 0,809 i 0,893, a średnie podobieństwo do pozostałych zdań 0,624, 0,662 i 0,731. Korelacja Spearmana między liczbą tokenów a średnim cosinusem wynosi 0,588. Jednocześnie maleje norma wektora (30,3; 28,6; 27,8). Wynika to ze sposobu tworzenia wektora zdania: uśredniane są wektory słów, więc w długim zdaniu poszczególne wkłady się znoszą, a wynik zbliża się do przeciętnego wektora korpusu. Krótkie zdania mają większa normę wektora, większa odległość od środka).  W USE norma jest stała (1,0), więc nie niesie informacji, ale efekt długości również jest widoczny, tylko słabszy: cosinus do centroidu wynosi 0,147, 0,214 i 0,235, a Spearman 0,318. W TF-IDF zachowanie jest podobne jak w fastText (Spearman 0,589), choć norma jest prawie stała (0,755, 0,739, 0,740). Wniosek praktyczny: wysokie wartości cosinusa, zwłaszcza w fastText, mogą wynikać z samej długości zdań, a nie z podobnego sensu, dlatego przy porównywaniu zdań należy uwzględniać ich długość. Wartości bezwzględne między modelami nie są porównywalne, ponieważ USE ma ogólnie niższą skalę (średnie podobieństwo do innych zdań około 0,02–0,04).

**5. Czy osadzenia zdań nadają się do wybranego zadania (anotacja wydźwięku)?**
_Odpowiedź:_ Nie. Sprawdzone zostało to klasyfikatorem (regresja logistyczna, 5-krotna walidacja krzyżowa) uczonym na samych osadzeniach, dla czterech klas: POZ, NEG, NEU i AMB. Dla uzgodnionych etykiet final accuracy wynosi 0,505 w fastText, 0,520 w USE i 0,490 w TF-IDF, przy bazie większościowej 0,415 (klasyfikator zawsze wybierający NEU). Przewaga nad bazą to więc tylko 0,09, 0,10 i 0,08. Macro-F1 jest niskie (0,322, 0,330 i 0,318), a czystość 10 najbliższych sąsiadów (0,321, 0,334 i 0,335) niewiele przewyższa poziom losowy (0,305). Wartość silhouette jest ujemna w fastText (−0,123) i bliska zera w USE (−0,013) oraz TF-IDF (−0,008), co oznacza, że zdania nie grupują się według wydźwięku.

Macierz pomyłek pokazuje, skąd bierze się wynik. Klasyfikator wybiera głównie klasę NEU (czułość 0,92 w fastText i 0,88 w USE), natomiast klasę NEG rozpoznaje rzadko (czułość 0,15 i 0,09), a klasy AMB wcale (0 w obu modelach; w zbiorze jest tylko 17 takich zdań). POZ rozpoznawane jest umiarkowanie (0,33 w fastText i 0,50 w USE). Dla etykiet gold, które mają po 50 zdań na klasę (baza 0,25), USE wypada lepiej niż fastText (accuracy 0,465 wobec 0,395, macro-F1 0,464 wobec 0,388), ale nadal są to niskie wartości. Wyniki dla final, A1 i A2 są do siebie zbliżone, więc wybór zestawu etykiet zmienia je mało. Wyższe accuracy dla final, A1 i A2 niż dla gold wynika głównie z przewagi klasy NEU w tych zbiorach.

Modele kodują raczej temat i słownictwo niż wydźwięk (odpowiedzi 1–2), więc nie nadają się do samodzielnej anotacji wydźwięku zdań. Mogłyby ewentualnie służyć jako cecha pomocnicza. Przy 200 zdaniach i bez przedziałów ufności wyniki są orientacyjne. 